# XD2-Net — 03: Evaluation

Loads trained models, runs test set evaluation, compares against a
Logistic Regression baseline trained on the same aggregated (exp) features.

**Prerequisite:** run notebooks 01 and 02 first.

In [ ]:
import os, sys, glob

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = '/content/drive/MyDrive/xd2-net'
    %pip install -e {REPO_ROOT} --quiet
else:
    REPO_ROOT = os.path.abspath('..')
    sys.path.insert(0, REPO_ROOT)

os.chdir(REPO_ROOT)
print('REPO_ROOT:', REPO_ROOT)

In [ ]:
import pickle
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression

from xd2net.config import load_config
from xd2net.model.xd2net import XD2Net
from xd2net.data.dataset import make_test_loader
from xd2net.training.metrics import evaluate_predictions
from xd2net.explain.explainer import Explainer

cfg = load_config('configs/bpic2012.yaml')
artifacts_dir = os.path.join(cfg.output.workspace_dir, 'artifacts')
ckpt_dir = os.path.join(cfg.output.workspace_dir, 'checkpoints')

In [ ]:
def load_best_checkpoint(bucket_key: str, model: XD2Net) -> XD2Net:
    """Load the checkpoint with the lowest val_loss for this bucket."""
    pattern = os.path.join(ckpt_dir, f'model_{bucket_key}_*.pt')
    checkpoints = glob.glob(pattern)
    if not checkpoints:
        raise FileNotFoundError(f'No checkpoint found for bucket {bucket_key} in {ckpt_dir}')
    # Pick the one with smallest val_loss encoded in filename
    best = min(checkpoints, key=lambda p: float(p.split('vl')[-1].replace('.pt', '')))
    model.load_state_dict(torch.load(best, map_location='cpu'))
    print(f'  Loaded: {os.path.basename(best)}')
    return model

In [ ]:
xd2_results = []
lr_results   = []

for n_size in cfg.data.prefix_lengths:
    bucket_key = f'1_{n_size}'
    art_path = os.path.join(artifacts_dir, f'bucket_{bucket_key}.pkl')
    if not os.path.exists(art_path):
        continue

    with open(art_path, 'rb') as f:
        art = pickle.load(f)

    print(f'\nBucket {bucket_key}')

    # ── XD2-Net evaluation ────────────────────────────────────────────────────
    model = XD2Net(art['dim_dynamic'], art['dim_static'], art['exp_dim'], art['max_len'], cfg.model)
    model = load_best_checkpoint(bucket_key, model)

    test_loader = make_test_loader(
        art['inputs_test'], art['outputs_test'], art['lengths_test'],
        batch_size=cfg.training.batch_size,
    )
    explainer = Explainer(model, device=cfg.training.device)
    df_pred, _ = explainer.explain_loader(
        test_loader, art['caseids_test'], art['feature_names'], bucket=bucket_key
    )

    m = evaluate_predictions(df_pred['GROUND_TRUTH'].values, df_pred['PREDICTION_PROB'].values,
                             target_names=['A_DECLINED', 'A_APPROVED'])
    xd2_results.append({'bucket': bucket_key, 'accuracy': m['accuracy'],
                        'precision': m['precision'], 'recall': m['recall'],
                        'f1': m['f1'], 'roc_auc': m['roc_auc']})
    print(f'  XD2-Net  acc={m["accuracy"]:.3f}  auc={m["roc_auc"]:.3f}  f1={m["f1"]:.3f}')

    # ── Logistic Regression baseline (on exp features) ────────────────────────
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(art['inputs_train']['input_exp'], art['outputs_train']['act_output'])
    lr_prob = lr.predict_proba(art['inputs_test']['input_exp'])[:, 1]
    ml = evaluate_predictions(art['outputs_test']['act_output'], lr_prob,
                              target_names=['A_DECLINED', 'A_APPROVED'])
    lr_results.append({'bucket': bucket_key, 'accuracy': ml['accuracy'],
                       'precision': ml['precision'], 'recall': ml['recall'],
                       'f1': ml['f1'], 'roc_auc': ml['roc_auc']})
    print(f'  LR       acc={ml["accuracy"]:.3f}  auc={ml["roc_auc"]:.3f}  f1={ml["f1"]:.3f}')

In [ ]:
# ── Side-by-side comparison table ────────────────────────────────────────────
df_xd2 = pd.DataFrame(xd2_results).set_index('bucket').add_prefix('xd2_')
df_lr  = pd.DataFrame(lr_results).set_index('bucket').add_prefix('lr_')
df_cmp = df_xd2.join(df_lr)
print(df_cmp.round(3).to_string())

In [ ]:
# ── Save results ──────────────────────────────────────────────────────────────
out_dir = os.path.join(cfg.output.workspace_dir, 'results')
os.makedirs(out_dir, exist_ok=True)
df_cmp.to_csv(os.path.join(out_dir, 'comparison_results.csv'))
print('Saved to', out_dir)